### 매물의 집객시설 수 정리

In [31]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import nearest_points

gdf_commercial = gpd.read_file(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\상권_행정구역_구분\서울시 상권분석서비스(영역-상권).shp')
gdf_commercial = gdf_commercial.to_crs(epsg=4326) # 좌표계 통일
# ['TRDAR_SE_C', 'TRDAR_SE_1', 'TRDAR_CD', 'TRDAR_CD_N', 'XCNTS_VALU', 'YDNTS_VALU', 'SIGNGU_CD', 'SIGNGU_CD_', 'ADSTRD_CD', 'ADSTRD_CD_', 'RELM_AR', 'geometry'] 상권 영역 내의 칼럼. geometry => 폴리곤
# 최상위 폴더 경로
root_folder = r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\3. 서울시_상가_분석데이터'
# 모든 csv 파일 경로를 저장할 리스트
csv_files = []

# root_folder 안의 폴더만 순회 (1단계 하위 폴더)
for folder_name in os.listdir(root_folder):
    subfolder_path = os.path.join(root_folder, folder_name)
    if os.path.isdir(subfolder_path):
        for file_name in os.listdir(subfolder_path):
            if file_name.lower().endswith('.csv'):
                file_path = os.path.join(subfolder_path, file_name)
                csv_files.append(file_path)
# CSV 파일들을 데이터프레임으로 읽기
dataframes = [pd.read_csv(file) for file in csv_files]

# 하나의 데이터프레임으로 합치기
combined_df = pd.concat(dataframes, ignore_index=True)
# 결과 확인
# print(f"총 {len(csv_files)}개의 CSV 파일을 불러왔습니다.")
# print(combined_df.head())
print(combined_df['articleNo'].value_counts())

# 상권 매핑
cols = ['articleNo', 'articleName','tradeTypeName' ,'latitude', 'longitude'] # combined_df 내의 칼럼 (경도, 위도)
df = combined_df[cols].copy()
df['geometry'] = df.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
gdf_points = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326') # 좌표계 통일
# 공간조인: 각 점이 포함된 상권 정보 붙이기
gdf_joined = gpd.sjoin(gdf_points, gdf_commercial, how='left', predicate='within')

# 상권의 centroid 계산
gdf_commercial = gdf_commercial.to_crs(epsg=5181)
gdf_commercial['centroid'] = gdf_commercial.geometry.centroid

# 중복값 처리: 가장 가까운 상권의 centroid 선택
closest_rows = []
for article_no in gdf_joined['articleNo'].unique():
    group = gdf_joined[gdf_joined['articleNo'] == article_no]
    
    if len(group) > 1:
        # 여러 개의 상권이 겹치는 경우, 가장 가까운 상권의 centroid를 찾음
        point = group.iloc[0]['geometry']  # 해당 상가의 위치
        group['distance'] = group['centroid'].apply(lambda x: x.distance(point))  # 거리 계산
        nearest_idx = group['distance'].idxmin()  # 가장 가까운 상권의 인덱스
        closest_rows.append(group.loc[nearest_idx])  # 가장 가까운 상권으로 선택
    else:
        closest_rows.append(group.iloc[0])  # 중복 없는 경우 그대로 추가

# 가장 가까운 상권으로 선택된 매물들로 GeoDataFrame 생성
gdf_closest = pd.concat(closest_rows, axis=1).T
gdf_closest = gpd.GeoDataFrame(gdf_closest, geometry='geometry', crs='EPSG:4326')

# 결과 확인
print(f"가장 가까운 상권 매핑 결과:\n{gdf_closest[['articleNo', 'articleName', 'tradeTypeName', 'TRDAR_CD']].head()}")
# 결과 확인
print(gdf_joined[['articleNo', 'articleName', 'tradeTypeName', 'TRDAR_CD']].head())
# 중복값 확인 
print(gdf_joined['articleNo'].value_counts())

articleNo
2518818058    1
2524638284    1
2524618474    1
2524647813    1
2524618524    1
             ..
2524100869    1
2524281894    1
2524289230    1
2524271539    1
2524043984    1
Name: count, Length: 98015, dtype: int64


KeyError: 'centroid'

In [36]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import nearest_points

# 상권 데이터 로드
gdf_commercial = gpd.read_file(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\상권_행정구역_구분\서울시 상권분석서비스(영역-상권).shp')
gdf_commercial = gdf_commercial.to_crs(epsg=4326)  # 좌표계 통일

# 최상위 폴더 경로
root_folder = r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\3. 서울시_상가_분석데이터'
csv_files = []

# CSV 파일 경로를 리스트에 추가
for folder_name in os.listdir(root_folder):
    subfolder_path = os.path.join(root_folder, folder_name)
    if os.path.isdir(subfolder_path):
        for file_name in os.listdir(subfolder_path):
            if file_name.lower().endswith('.csv'):
                file_path = os.path.join(subfolder_path, file_name)
                csv_files.append(file_path)

# CSV 파일들을 데이터프레임으로 읽기
dataframes = [pd.read_csv(file) for file in csv_files]

# 모든 CSV를 하나의 데이터프레임으로 합치기
combined_df = pd.concat(dataframes, ignore_index=True)

# 상권 매핑
cols = ['articleNo', 'articleName', 'tradeTypeName', 'latitude', 'longitude']  # 원하는 칼럼
df = combined_df[cols].copy()
df['geometry'] = df.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
gdf_points = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# 공간조인: 각 점이 포함된 상권 정보 붙이기
gdf_joined = gpd.sjoin(gdf_points, gdf_commercial, how='left', predicate='within')

# 상권의 centroid 계산
gdf_commercial = gdf_commercial.to_crs(epsg=5181)
gdf_commercial['centroid'] = gdf_commercial.geometry.centroid

# 중복값 처리: 가장 가까운 상권의 centroid 선택
closest_rows = []
for article_no in gdf_joined['articleNo'].unique():
    group = gdf_joined[gdf_joined['articleNo'] == article_no]
    
    if len(group) > 1:
        # 여러 개의 상권이 겹치는 경우, 가장 가까운 상권의 centroid를 찾음
        point = group.iloc[0]['geometry']  # 해당 상가의 위치
        # 가장 가까운 상권을 찾기 위해 centroid와의 거리를 계산
        gdf_commercial['distance'] = gdf_commercial['centroid'].apply(lambda x: point.distance(x)) 
        nearest_idx = gdf_commercial['distance'].idxmin()  # 가장 가까운 상권의 인덱스
        closest_rows.append(gdf_commercial.loc[nearest_idx])  # 가장 가까운 상권으로 선택
    else:
        closest_rows.append(group.iloc[0])  # 중복 없는 경우 그대로 추가

# 가장 가까운 상권으로 선택된 매물들로 GeoDataFrame 생성
gdf_closest = pd.concat(closest_rows, axis=1).T
gdf_closest = gpd.GeoDataFrame(gdf_closest, geometry='geometry', crs='EPSG:4326')

# 결과 확인
print(f"가장 가까운 상권 매핑 결과:\n{gdf_closest[['articleNo', 'articleName', 'tradeTypeName', 'TRDAR_CD']].head()}")
# 결과 확인
print(gdf_closest[['articleNo', 'articleName', 'tradeTypeName', 'TRDAR_CD']].head())
# 중복값 확인
print(gdf_closest['articleNo'].value_counts())

# 필요한 칼럼만 추출
output_cols = ['articleNo', 'articleName', 'tradeTypeName', 'TRDAR_CD']
final_df = gdf_closest[output_cols].copy()

# 파일 저장 (경로는 원하는 대로 수정 가능)
output_path = r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\매물_상권_매핑_데이터.csv'
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"최종 파일이 저장되었습니다: {output_path}")

가장 가까운 상권 매핑 결과:
    articleNo articleName tradeTypeName TRDAR_CD
0  2524638284       단지내상가            매매      NaN
1  2524618474        일반상가            월세  3110994
2  2524647813        일반상가            월세  3130310
3  2524618524        일반상가            월세  3110994
4  2524668187        일반상가            월세  3110984
    articleNo articleName tradeTypeName TRDAR_CD
0  2524638284       단지내상가            매매      NaN
1  2524618474        일반상가            월세  3110994
2  2524647813        일반상가            월세  3130310
3  2524618524        일반상가            월세  3110994
4  2524668187        일반상가            월세  3110984
articleNo
2518818058    1
2524638284    1
2524618474    1
2524647813    1
2524618524    1
             ..
2524426823    1
2524427927    1
2524558166    1
2524582883    1
2524512540    1
Name: count, Length: 96302, dtype: int64
최종 파일이 저장되었습니다: C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\매물_상권_매핑_데이터.csv


In [52]:
# 집객시설 데이터와 매물 상권 맵핑 데이터 붙이기
import pandas as pd
df_commercial = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\매물_상권_매핑_데이터.csv")
df_commercial.rename(columns={'TRDAR_CD':'상권_코드'}, inplace=True)
print(df_commercial['articleNo'].value_counts())
df_people = pd.read_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\서울시 상권분석서비스(집객시설-상권).csv', encoding = 'euc-kr')
df_people['기준_년분기_코드'] = df_people['기준_년분기_코드'].astype(str)
df_people = df_people[df_people['기준_년분기_코드'] == '20244']
df_merged = df_commercial.merge(df_people[['상권_코드','집객시설_수', '관공서_수', '은행_수', '종합병원_수', '일반_병원_수', '약국_수', '유치원_수', '초등학교_수', '중학교_수', '고등학교_수', '대학교_수', '백화점_수', '슈퍼마켓_수', '극장_수', '숙박_시설_수', '지하철_역_수', '버스_정거장_수']], on = '상권_코드', how = 'left')
df_merged.to_csv(r'C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\매물_집객시설.csv', index = False, encoding='utf-8-sig')

articleNo
2.518818e+09    1
2.524638e+09    1
2.524618e+09    1
2.524648e+09    1
2.524619e+09    1
               ..
2.524427e+09    1
2.524428e+09    1
2.524558e+09    1
2.524583e+09    1
2.524513e+09    1
Name: count, Length: 96302, dtype: int64


In [ ]:
# 전체 통합 데이터와 매핑
import pandas as pd
people = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\매물_집객시설.csv")
total = pd.read_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\매물_피처병합.csv")
merged = people.merge(total, on = ['articleNo','articleName','tradeTypeName'], how = 'left')
print(merged.columns.tolist())
merged = merged[[
    # 영어 컬럼 우선
    'articleNo', 'articleName', 'tradeTypeName', 'floorInfo',
    'rentPrc', 'dealOrWarrantPrc', 'area1', 'area2', 'direction',
    'articleConfirmYmd', 'tagList', 'latitude', 'longitude',
    
    # 행정구역 정보
    '구', '동', '상권_코드',

    # 집객시설 관련
    '집객시설_수', '관공서_수', '은행_수', '종합병원_수', '일반_병원_수', '약국_수',
    '유치원_수', '초등학교_수', '중학교_수', '고등학교_수', '대학교_수',
    '백화점_수', '슈퍼마켓_수', '극장_수', '숙박_시설_수', '지하철_역_수', '버스_정거장_수',

    # 유동인구 정보
    '평균_평일_유동인구_수', '평균_주말_유동인구_수', '평일_대비_주말_유동인구_비율',
    'est_총_유동인구_수', 'est_시간대_00_06_유동인구_수', 'est_시간대_06_11_유동인구_수',
    'est_시간대_11_14_유동인구_수', 'est_시간대_14_17_유동인구_수',
    'est_시간대_17_21_유동인구_수', 'est_시간대_21_24_유동인구_수',

    # 건물 및 도로 특성
    '반경500m내_카페수', '접한길이_m', '도로2m이내접착길이_m', '교차로_존재_여부',

    # 교통 접근성
    '지하철역_거리(m)', '버스정류장_거리(m)'
]]
merged.to_csv(r"C:\Users\iq750\bootcamp_git\Final-Project-2_Team1\data\매물추천\매물_피처병합.csv", encoding='utf-8-sig', index = False)

['articleNo', 'articleName', 'tradeTypeName', '상권_코드', '집객시설_수', '관공서_수', '은행_수', '종합병원_수', '일반_병원_수', '약국_수', '유치원_수', '초등학교_수', '중학교_수', '고등학교_수', '대학교_수', '백화점_수', '슈퍼마켓_수', '극장_수', '숙박_시설_수', '지하철_역_수', '버스_정거장_수', 'floorInfo', 'rentPrc', 'dealOrWarrantPrc', 'area1', 'area2', 'direction', 'articleConfirmYmd', 'tagList', 'latitude', 'longitude', '구', '동', '지하철역_거리(m)', '버스정류장_거리(m)', '평균_평일_유동인구_수', '평균_주말_유동인구_수', '평일_대비_주말_유동인구_비율', 'est_총_유동인구_수', 'est_시간대_00_06_유동인구_수', 'est_시간대_06_11_유동인구_수', 'est_시간대_11_14_유동인구_수', 'est_시간대_14_17_유동인구_수', 'est_시간대_17_21_유동인구_수', 'est_시간대_21_24_유동인구_수', '반경500m내_카페수', '접한길이_m', '도로2m이내접착길이_m', '교차로_존재_여부']


: 